# Task 4 - Step 1: Train Vanilla, GCSC and PROSER

| Config | Objective | Schedule |
|---|---|---|
| `vanilla` | cross-entropy, crop+flip | SGD 0.1, momentum 0.9, wd 5e-4, cosine, batch 128, 100 epochs |
| `gcsc` | the same with `RandAugment(num_ops=2, magnitude=9)` after crop/flip | identical |
| `proser` | classifier + data placeholders (β=1, γ=0.1), initialised from the selected Vanilla checkpoint | SGD 1e-3, cosine, batch 128, 50 epochs |

Checkpoint rule for all three: highest CIFAR-10 **validation** accuracy. **No CIFAR-100 image is visible here.**
Finished runs are skipped, never overwritten; `TASK4_RUNS=vanilla` trains a subset.

In [1]:
# ---- Task 4 common header (identical in every Task 4 notebook) ----
# NOTE: no CIFAR-100 image is decoded or loaded anywhere except notebook 03, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "task4" / "training.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.config import load_config
from task4.data import cifar10
from task4.models.resnet_cifar import build_model

T4 = REPO / "task4"
CFG_DIR = T4 / "configs"
SEED = 6304
# Smoke mode (env TASK4_SMOKE=1): 2 epochs on 2000 training images; outputs under _smoke/; notebook 03 uses
# RANDOM stand-in "unknown" images, so no CIFAR-100 image is touched.
SMOKE = os.environ.get("TASK4_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T4 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T4 / "results" / "tables"       # data-preparation files (same path in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T4 / "checkpoints" / SUB            # git-ignored
CACHE = T4 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUNS = ["vanilla", "gcsc", "proser"]
LABEL = {"vanilla": "Vanilla", "gcsc": "GCSC", "proser": "PROSER"}
SCORES = ["msp", "mls", "energy", "mahalanobis"]         # post-hoc scores on the frozen Vanilla model
SCORE_LABEL = {"msp": "MSP", "mls": "MLS", "energy": "Energy", "mahalanobis": "Mahalanobis",
               "placeholder": "PROSER placeholder"}


def run_dir(n):
    return CKPT / load_config(CFG_DIR, n)["run_name"]


def load_trained(n):
    """Frozen model with the selected checkpoint of run ``n``."""
    cfg = load_config(CFG_DIR, n)
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(cfg["model"]["num_classes"], cfg["model"]["dummy_classifiers"], cfg["seed"])
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck, cfg


@torch.no_grad()
def extract(model, x_u8, batch_size=512):
    """Penultimate features and logits of a frozen model (no augmentation)."""
    feats, logits = [], []
    for s in range(0, len(x_u8), batch_size):
        f, z = model(cifar10.normalize(x_u8[s:s + batch_size].to(DEVICE)))
        feats.append(f.float().cpu()); logits.append(z.float().cpu())
    return torch.cat(feats), torch.cat(logits)


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
from task4.methods import build_method
from task4.training import train

data = {"train": cifar10.load("train"), "val": cifar10.load("val")}
if SMOKE:
    data = {k: (v[0][:2000], v[1][:2000]) for k, v in data.items()}
print({k: len(v[1]) for k, v in data.items()})


def init_proser_from_vanilla(model, vanilla_ckpt, num_known=10):
    """Copy every Vanilla parameter; the classifier keeps its 10 known rows and adds randomly initialised dummies."""
    sd = torch.load(vanilla_ckpt, map_location="cpu", weights_only=False)["model"]
    own = model.state_dict()
    for k, v in sd.items():
        if own[k].shape == v.shape:
            own[k] = v
        else:                                     # net.fc.weight / net.fc.bias: [10+D, ...] vs [10, ...]
            own[k][:num_known] = v
    model.load_state_dict(own)
    return model

{'train': 45000, 'val': 5000}


In [3]:
runs = os.environ.get("TASK4_RUNS", ",".join(RUNS)).split(",")
SUMMARIES = {}
for name in runs:
    cfg = load_config(CFG_DIR, name)
    if SMOKE:
        cfg["train"]["epochs"] = 2
    rd = CKPT / cfg["run_name"]
    if (rd / "summary.json").exists():
        print(f"skip {name}: finished run already in {rd}")
        SUMMARIES[name] = json.loads((rd / "summary.json").read_text())
        continue
    print(f"\n=== {name} ({cfg['method']}, {cfg['train']['epochs']} epochs) ===")
    model = build_model(cfg["model"]["num_classes"], cfg["model"]["dummy_classifiers"], cfg["seed"])
    if cfg.get("init_from"):
        src = CKPT / load_config(CFG_DIR, cfg["init_from"])["run_name"] / "best.pt"
        assert src.exists(), f"{name} needs the {cfg['init_from']} checkpoint first"
        model = init_proser_from_vanilla(model, src, cfg["model"]["num_classes"])
        print(f"initialised from {src.relative_to(REPO)}")
    SUMMARIES[name] = train(model, build_method(cfg), data, cfg, rd, DEVICE)
    del model
    torch.cuda.empty_cache()


=== vanilla (vanilla, 100 epochs) ===


[vanilla] epoch   1/100  cls_loss=1.9260  train_acc=0.3129  val_acc=0.4340  *  (27s)


[vanilla] epoch   2/100  cls_loss=1.4045  train_acc=0.4874  val_acc=0.5152  *  (27s)


[vanilla] epoch   3/100  cls_loss=1.1076  train_acc=0.6040  val_acc=0.6178  *  (28s)


[vanilla] epoch   4/100  cls_loss=0.8850  train_acc=0.6901  val_acc=0.7212  *  (28s)


...

[proser] epoch  42/50  total_loss=0.0125  clf_placeholder_loss=0.0109  l1=0.0065  l2=0.0045  data_placeholder_loss=0.0155  train_acc=0.9998  mixed_dummy_wins=1.0000  val_acc=0.9222  (28s)


[proser] epoch  44/50  total_loss=0.0130  clf_placeholder_loss=0.0115  l1=0.0070  l2=0.0045  data_placeholder_loss=0.0153  train_acc=0.9995  mixed_dummy_wins=1.0000  val_acc=0.9222  (28s)


[proser] epoch  46/50  total_loss=0.0122  clf_placeholder_loss=0.0107  l1=0.0065  l2=0.0042  data_placeholder_loss=0.0152  train_acc=0.9998  mixed_dummy_wins=1.0000  val_acc=0.9284  (28s)


[proser] epoch  48/50  total_loss=0.0117  clf_placeholder_loss=0.0101  l1=0.0059  l2=0.0042  data_placeholder_loss=0.0157  train_acc=0.9999  mixed_dummy_wins=1.0000  val_acc=0.9226  (28s)


[proser] epoch  50/50  total_loss=0.0131  clf_placeholder_loss=0.0116  l1=0.0070  l2=0.0046  data_placeholder_loss=0.0153  train_acc=0.9996  mixed_dummy_wins=1.0000  val_acc=0.9298  (28s)


In [4]:
rows = []
for n in RUNS:
    p = CKPT / load_config(CFG_DIR, n)["run_name"] / "summary.json"
    if p.exists():
        s = json.loads(p.read_text())
        rows.append({"config": n, "method": LABEL[n], "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"],
                     "best_val_acc": s["best_val_acc"], "train_minutes": s["train_seconds"] / 60,
                     "uses_cifar100": s["uses_cifar100"]})
TRAIN_SUMMARY = pd.DataFrame(rows)
TRAIN_SUMMARY.to_csv(TAB / "task4_training_summary.csv", index=False)
TRAIN_SUMMARY.round(4)

,config,method,best_epoch,epochs_run,best_val_acc,train_minutes,uses_cifar100
0,vanilla,Vanilla,95,100,0.9526,47.2654,False
1,gcsc,GCSC,99,100,0.9568,99.7472,False
2,proser,PROSER,1,50,0.9486,23.2395,False
